In [2]:
!pip install rdkit-pypi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 6.0 MB/s eta 0:00:0000:0100:01


In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np

In [26]:
import requests 
import pubchempy as pcp

def extract_smiles_from_record(compound_dict):
    props = compound_dict.get('record', {}).get('props', [])
    smiles = []
    for p in props:
        urn = p.get('urn', {})
        if urn.get('label') == 'SMILES' and 'sval' in p.get('value', {}):
            smiles.append(p['value']['sval'])
    if len(smiles) >= 1:
        return smiles[0]
    elif len(smiles) == 0:
        return None



def find_with_pubchempy(formula):
    print('loooking formula ', formula)
    try:
        compounds = pcp.get_compounds(formula, "formula",listkey_count=1)
        names = [compound.to_dict()['iupac_name']for compound in compounds] 
        smiles = [
            compound.to_dict()['canonical_smiles'] 
            if compound.to_dict()['canonical_smiles'] is not None 
            else extract_smiles_from_record(compound.to_dict()) 
            for compound in compounds
        ] 
        print('names', names, 'smiles', smiles)
        return names, smiles
    except Exception as e:
        print('we have hit the exception: ', e)
        return [], []


# print(find_with_pubchempy("C2H6O"))  # ethanol / dimethyl ether, etc.


In [29]:
df = pd.read_csv("morgan_fingerprints_training.csv")
df

,Unnamed: 0,Formula,Num_Runs,Run_Names,Avg_Score
0,15289,C9H17N3O5S,13,"large, large, large, large, large, large, larg...",74.270769
1,6012,C35H52O4,9,"bigmed, large, large, large, large, large, lar...",85.707778
2,7376,C40H64O6,8,"bigmed, large, large, large, large, large, med...",74.047500
3,4295,C28H57O7P,8,"bigmed, large, large, large, large, large, med...",72.431250
4,6092,C35H58O5,8,"bigmed, large, large, large, large, large, med...",75.741250
...,...,...,...,...,...
1995,740,C14H27NO3,3,"bigmed, med, tiny",69.443333
1996,9535,C47H85O7P,3,"bigmed, large, tiny",58.556667
1997,7214,C39H72O6,3,"large, large, med",54.190000
1998,9553,C47H87O18P,3,"bigmed, med, tiny",55.476667


In [ ]:
import pandas as pd 
import time 
from tqdm import tqdm

# Add new columns if they don't exist
if 'name' not in df.columns:
    df['name'] = None
if 'smiles' not in df.columns:
    df['smiles'] = None

# Loop through rows
for idx, row in tqdm(df.iterrows(), total=len(df)):
    if pd.notnull(row.get('smiles')) and row['smiles']:
        print('we are skipping b/c already populated')
        # skip if already populated
        continue
    
    formula = row['Formula'] if 'Formula' in df.columns else None
    if not formula:
        continue

    name, smiles = find_with_pubchempy(formula)
    df.at[idx, 'name'] = name
    df.at[idx, 'smiles'] = smiles

    # Save progress every 50 entries
    if idx % 10 == 0:
        df.to_csv("morgan_fingerprints_training_progress.csv", index=False)
        print(f"Progress saved at row {idx}")
    
    # polite delay for PubChem servers
    time.sleep(0.5)

# Final save
df.to_csv("morgan_fingerprints_training_progress.csv", index=False)
print("✅ Done! Saved as morgan_fingerprints_training_progress.csv")


  0%|          | 0/2000 [00:00<?, ?it/s]

loooking formula  C9H17N3O5S
names ['[(2R)-2-amino-3-sulfanylpropanoyl] (2S)-2-amino-3-[(2S)-2-aminopropanoyl]oxypropanoate'] smiles ['C[C@@H](C(=O)OC[C@@H](C(=O)OC(=O)[C@H](CS)N)N)N']
Progress saved at row 0


  0%|          | 1/2000 [00:04<2:26:27,  4.40s/it]

loooking formula  C35H52O4
names ['(1R,5R,7S,8R)-4-hydroxy-8-methyl-3,5,7-tris(3-methylbut-2-enyl)-8-(4-methylpent-3-enyl)-1-(2-methylpropanoyl)bicyclo[3.3.1]non-3-ene-2,9-dione'] smiles ['CC(C)C(=O)[C@]12C(=O)C(=C([C@](C1=O)(C[C@@H]([C@@]2(C)CCC=C(C)C)CC=C(C)C)CC=C(C)C)O)CC=C(C)C']


  0%|          | 2/2000 [00:08<2:19:45,  4.20s/it]

...

loooking formula  C16H26N6O5
names ['(2S)-2-[[(2S)-2-[(2-acetamidoacetyl)amino]-3-(1H-imidazol-5-yl)propanoyl]amino]-6-aminohexanoic acid'] smiles ['CC(=O)NCC(=O)N[C@@H](CC1=CN=CN1)C(=O)N[C@@H](CCCCN)C(=O)O']


  2%|▏         | 38/2000 [02:47<2:19:17,  4.26s/it]

loooking formula  C32H62NO11P
names ['[2-[hydroxy-[2-[[2-(3-methoxy-3-methylbutoxy)-2-methylpropanoyl]amino]ethoxy]phosphoryl]oxy-3-(2,2,4-trimethylpentanoyloxy)propyl] 2,2,4,4-tetramethylpentanoate'] smiles ['CC(C)CC(C)(C)C(=O)OCC(COC(=O)C(C)(C)CC(C)(C)C)OP(=O)(O)OCCNC(=O)C(C)(C)OCCC(C)(C)OC']


  2%|▏         | 39/2000 [02:52<2:25:32,  4.45s/it]

loooking formula  C37H59NO8


In [ ]:
# # Suppose you have a DataFrame of SMILES strings
# df = pd.DataFrame({
#     'smiles': ['CCO', 'CCN', 'c1ccccc1O']
# })

# # Convert to RDKit molecules
# mols = [Chem.MolFromSmiles(s) for s in df['smiles']]

# # Compute Morgan fingerprints
# fps = [AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=1024) for m in mols]

# # Convert to numpy array (optional, for ML)
# fps_array = np.array([np.frombuffer(fp.ToBitString().encode('utf-8'), 'u1') - ord('0') for fp in fps])
# # 